# Week 3: Entity Extraction

This notebook extracts structured facts from Week 2 cleaned listing remarks and prepares a transparent evaluation workflow.


In [9]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.entity_extraction import EntityExtractor

entities_path = ROOT / 'data/processed/listing_sample_entities.csv'
df = pd.read_csv(entities_path)
extractor = EntityExtractor()
print(f'Loaded {len(df):,} listings')
df.shape


Loaded 1,000 listings


(1000, 15)

## Extracted fields

The extractor uses regular expressions for bedrooms, bathrooms, square footage, and dollar-denominated prices. It uses the Week 1 taxonomy for amenity, interior-feature, and exterior-feature matching.


In [10]:
entity_columns = [
    'bedrooms_extracted', 'bathrooms_extracted', 'square_feet_extracted',
    'price_mentioned_extracted', 'amenities_extracted',
    'interior_features_extracted', 'exterior_features_extracted',
]

coverage = pd.DataFrame({
    'entity': entity_columns,
    'listings_with_value': [
        df[column].notna().sum() if column.endswith('_extracted') and 'features' not in column and 'amenities' not in column else (df[column].fillna('[]') != '[]').sum()
        for column in entity_columns
    ],
})
coverage


,entity,listings_with_value
0,bedrooms_extracted,758
1,bathrooms_extracted,609
2,square_feet_extracted,375
3,price_mentioned_extracted,72
4,amenities_extracted,520
5,interior_features_extracted,811
6,exterior_features_extracted,761


In [11]:
display_columns = ['L_ListingID', 'L_City', 'remarks_cleaned'] + entity_columns
df[display_columns].head(5)


,L_ListingID,L_City,remarks_cleaned,bedrooms_extracted,bathrooms_extracted,square_feet_extracted,price_mentioned_extracted,amenities_extracted,interior_features_extracted,exterior_features_extracted
0,1157753806,San Bernardino,"Tenants in the front unit just moved out, and ...",3.0,1.0,1120.0,2300.0,[],[],[]
1,1177906910,Temecula,Tucked within one of De Luz's most coveted enc...,3.0,NaN,6000.0,NaN,"[""pool"", ""spa""]","[""custom cabinetry"", ""chef's kitchen"", ""walk-i...","[""pergola""]"
2,1190063789,Monterey,Spectacular Monterey Bay and city-light views ...,NaN,NaN,NaN,NaN,[],"[""vaulted ceilings"", ""breakfast bar"", ""tile fl...","[""two-car garage"", ""patio""]"
3,1189804557,Lake Elsinore,***HIGHLY UPGRADED POOL HOME + PAID OFF SOLAR*...,6.0,3.0,NaN,NaN,"[""pool"", ""gym"", ""spa""]","[""laminate flooring""]","[""covered patio"", ""front yard"", ""backyard"", ""p..."
4,1180192900,Los Angeles,Seller is offering an interest-rate buydown to...,3.0,2.5,1245.0,40000.0,"[""gated community"", ""pool""]","[""walk-in closet"", ""kitchen""]","[""attached garage""]"


## Annotation and evaluation

A random 250-listing annotation template is saved as `data/processed/week3_annotation_template.csv`. Fill only the `expected_*` columns after reading each remark. Use JSON lists for feature fields, for example `["pool", "spa"]`.

Do not copy the extracted fields into the expected fields. The expected fields are the manually verified reference answers used to calculate precision, recall, and F1.


In [12]:
annotation_path = ROOT / 'data/processed/week3_annotation_template.csv'
annotation_df = pd.read_csv(annotation_path)
expected_columns = [column for column in annotation_df.columns if column.startswith('expected_')]
reviewed_rows = annotation_df[expected_columns].notna().any(axis=1).sum()
print(f'Annotation rows: {len(annotation_df)}')
print(f'Rows with at least one manual answer: {reviewed_rows}')


Annotation rows: 250
Rows with at least one manual answer: 250


## Interpretation notes

- `price_mentioned_extracted` means the first explicit price in the remark. It may be a rent or historical price, so it requires manual review.
- A property can mention multiple units. The baseline extractor returns the first explicit bedroom, bathroom, and square-footage mention. Such cases should be documented in `review_notes`.
- After annotation, run `python scripts/evaluate_entities.py` from the project root to create `week3_evaluation_report.md`.


## Evaluation Error Audit: TP, TN, FP, and FN

The scalar evaluation uses **exact-value entity matching**:

- **TP (true positive):** the extracted value exactly matches the manual expected value.
- **TN (true negative):** both extracted and expected values are blank. TN is useful for inspection, but does **not** enter precision, recall, or F1.
- **FP (false positive):** the extractor outputs a value that is absent from the expected label or is the wrong value.
- **FN (false negative):** an expected value is missing or the extracted value is wrong.

When a scalar value is wrong (for example, predicted `3` but expected `3.5`), it counts as **both one FP and one FN**. This prevents an almost-correct number from being treated as fully correct.

### Checked examples from the 250 manually verified listings

| Entity | Case | Listing ID | Expected | Predicted | Why this is a useful check |
|---|---|---:|---:|---:|---|
| bedrooms | TP | 1090166117 | 3 | 3 | “single story 3-bedroom” is an explicit correct count. |
| bedrooms | TN | 1151236113 | blank | blank | No reliable total bedroom count appears in the chosen label. |
| bedrooms | FN + FP | 1015893587 | 3 | 2 | The remark describes two upstairs bedrooms plus a main bedroom/den; the extractor undercounts the total. |
| bathrooms | TP | 1090166117 | 3 | 3 | “3-bathroom retreat” is an explicit correct count. |
| bathrooms | TN | 1119482752 | blank | blank | “updated kitchen and bathrooms” is a generic mention, not a numeric total. |
| bathrooms | FP | 1150319693 | blank | 2 | In “2 bedrooms and bath”, `2` belongs to bedrooms, not bathrooms. |
| bathrooms | FN + FP | 1152452809 | 3.5 | 3 | “3 full bathrooms and 1 half bath” requires combining counts; `3` is not an exact match. |
| square_feet | TP | 1152452809 | 2330 | 2330 | “2330 square feet of living space” is the primary living-area figure. |
| square_feet | TN | 1015893587 | blank | blank | No square-footage value appears in this listing. |
| square_feet | FP | 1151236113 | blank | 200 | “200 square feet” describes a breezeway/office area, not the primary property size. |
| square_feet | FN + FP | 1159454962 | 943 | 430 | The expected “943 Living square feet” is missed; the extractor instead finds a later 430-square-foot addition. |
| price_mentioned | FP | 1169594162 | blank | 140 | `$140/month` is a solar-lease payment, not the listing price. |
| price_mentioned | FN | 1158525039 | 550000 | blank | “Now only 550000” is a listing-price statement without a dollar sign. |

The table below is generated from the current annotation CSV. Its assertions ensure that the displayed examples still have the stated classification after later extractor changes.


In [13]:
# Reproducible TP/TN/FP/FN audit examples
audit_cases = [
    ("bedrooms", "TP", 1090166117),
    ("bedrooms", "TN", 1151236113),
    ("bedrooms", "FN", 1015893587),
    ("bathrooms", "TP", 1090166117),
    ("bathrooms", "TN", 1119482752),
    ("bathrooms", "FP", 1150319693),
    ("bathrooms", "FN", 1152452809),
    ("square_feet", "TP", 1152452809),
    ("square_feet", "TN", 1015893587),
    ("square_feet", "FP", 1151236113),
    ("square_feet", "FN", 1159454962),
    ("price_mentioned", "FP", 1169594162),
    ("price_mentioned", "FN", 1158525039),
]

def scalar_case(predicted, expected):
    predicted = pd.to_numeric(pd.Series([predicted]), errors="coerce").iloc[0]
    expected = pd.to_numeric(pd.Series([expected]), errors="coerce").iloc[0]
    if pd.isna(predicted) and pd.isna(expected):
        return "TN"
    if pd.notna(predicted) and pd.notna(expected) and predicted == expected:
        return "TP"
    if pd.notna(predicted):
        return "FP" if pd.isna(expected) else "FP + FN"
    return "FN"

def excerpt(text, limit=240):
    return str(text).replace("\n", " ")[:limit] + "..."

audit_rows = []
for entity, requested_case, listing_id in audit_cases:
    row = annotation_df.loc[annotation_df["L_ListingID"] == listing_id].iloc[0]
    actual_case = scalar_case(row[f"{entity}_extracted"], row[f"expected_{entity}"])
    assert requested_case in actual_case, (entity, listing_id, requested_case, actual_case)
    audit_rows.append({
        "entity": entity,
        "requested_case": requested_case,
        "actual_case": actual_case,
        "L_ListingID": listing_id,
        "expected": row[f"expected_{entity}"],
        "predicted": row[f"{entity}_extracted"],
        "remark_excerpt": excerpt(row["remarks_cleaned"]),
    })

pd.DataFrame(audit_rows)


,entity,requested_case,actual_case,L_ListingID,expected,predicted,remark_excerpt
0,bedrooms,TP,TP,1090166117,3.0,3.0,"Turn-key licensed short-term rental asset, dee..."
1,bedrooms,TN,TN,1151236113,NaN,NaN,Custom-built country home on 5 peaceful acres....
2,bedrooms,FP,FP,1152364976,NaN,2.0,"Nestled in the exclusive, gated community of S..."
3,bathrooms,TP,TP,1090166117,3.0,3.0,"Turn-key licensed short-term rental asset, dee..."
4,bathrooms,TN,TN,1119482752,NaN,NaN,BACK IN THE MARKET Short Sale Opportunity! Cha...
5,bathrooms,FP,FP,1150319693,NaN,2.0,This HOMEPATH property is a Victorian style 3 ...
6,bathrooms,FN,FP + FN,1152452809,3.5,3.0,Beautiful and spacious mountain home at 42554 ...
7,square_feet,TP,TP,1152452809,2330.0,2330.0,Beautiful and spacious mountain home at 42554 ...
8,square_feet,TN,TN,1015893587,NaN,NaN,Spacious home in the desirable Top O'Topanga n...
9,square_feet,FP,FP,1151021879,NaN,11545.0,"Set for completion summer 2027, this exception..."


## Error Analysis: Top Failure Patterns

The remaining mistakes are not random; they cluster into a small number of recurring listing-language patterns.

1. **Multiple or component room counts.** A listing can describe a main home, a guest suite, and an ADU separately, or state “3 full bathrooms and 1 half bath.” A single scalar output may capture only one component instead of the intended total. This is the main remaining source of bedroom and bathroom mismatches.
2. **Non-primary square-footage figures.** Listings often mention lot size, garage space, deck area, an addition, or a guest house alongside the primary living area. The extractor now ranks local phrases such as `main residence` and `living space`, but compound and multi-unit listings can still be ambiguous.
3. **Amounts that are not the listing price.** Monthly homeowners-association dues, rent, lease payments, credits, and historical amounts may contain `$` but are not asking prices. Conversely, a listing can write `Now only 550000` without a dollar sign, which the current conservative price pattern misses.
4. **Gold-label scope decisions.** Whether to record the main residence, an ADU, total space, or every component must be consistent in the manual labels. Two reviewed labels were corrected during this audit because the original blank values conflicted with explicit text.

### Next improvements

- Add a compositional rule for patterns such as `3 full bathrooms and 1 half bath`.
- Add price-context rules that reject `per month`, `dues`, `lease`, and `credit`, while accepting labelled amounts such as `price 550000` or `now only 550000`.
- Keep reviewing multi-unit records separately so the annotation guideline explicitly states whether outputs represent the main residence or the whole property.
